# 第9章 文件内容操作

为了长期保存数据以便重复使用、修改和共享,必须将数据以文件的形式存储到外部存储介质(如磁盘、U盘等)中。

按文件中数据的组织形式把文件分为文本文件和二进制文件两类：
* **文本文件**: 存储的是常规字符串,由若干文本行组成,通常每行以换行符`\n`结尾。可以使用字处理软件(如记事本)进行编辑。
* **二进制文件**: 把对象内容以字节串(`bytes`)进行存储,通常无法被人类直接阅读和理解,需要专门的软件进行解码。如图像、音视频、可执行文件、Office文档等。

## 9.1 文件操作基本知识

### 9.1.1 内置函数open()
`open(file, mode='r', buffering=-1, encoding=None, errors=None, newline=None, closefd=True, opener=None)`

**文件打开模式:**
| 模式 | 说明 |
|---|---|
| `r` | 读模式(默认模式,可省略),如果文件不存在则抛出异常 |
| `w` | 写模式,如果文件已存在,先清空原有内容 |
| `x` | 写模式,创建新文件,如果文件已存在则抛出异常 |
| `a` | 追加模式,不覆盖文件中原有内容 |
| `b` | 二进制模式(可与其他模式组合使用) |
| `t` | 文本模式(默认模式,可省略) |
| `+` | 读、写模式(可与其他模式组合使用) |

In [ ]:
# f1 = open('file1.txt', 'r') # 以读模式打开文件
# f2 = open('file2.txt', 'w') # 以写模式打开文件
# f1.close() # 必须关闭文件对象

### 9.1.2 文件对象属性与常用方法

| 方法 | 功能说明 |
|---|---|
| `close()` | 把缓冲区的内容写入文件,同时关闭文件,并释放文件对象 |
| `flush()` | 把缓冲区的内容写入文件,但不关闭文件 |
| `read([size])` | 从文本文件中读取size个字符内容作为结果返回(省略则读取所有) |
| `readline()` | 从文本文件中读取一行内容作为结果返回 |
| `readlines()` | 把文本文件中的每行文本作为一个字符串存入列表中,返回该列表 |
| `seek(offset[, whence])`| 把文件指针移动到新的字节位置(0:文件头, 1:当前位置, 2:文件尾) |
| `tell()` | 返回文件指针的当前位置 |
| `write(s)` | 把s的内容写入文件 |
| `writelines(s)` | 把字符串列表写入文本文件,不添加换行符 |

### 9.1.3 上下文管理语句with
使用 `with` 可以自动管理资源，不论因为什么原因跳出 `with` 块，总能保证文件被正确关闭。

In [ ]:
# with 的基本语法
# with open('sample.txt', 'r') as fp:
#     content = fp.read()

# 支持同时打开多个文件
# with open('test.txt', 'r') as src, open('test_new.txt', 'w') as dst:
#     dst.write(src.read())

## 9.2 文本文件内容操作案例精选

In [ ]:
# 示例9-1 向文本文件中写入内容,然后再读出
s = 'Hello world\n文本文件的读取方法\n文本文件的写入方法\n'
with open('sample.txt', 'w', encoding='cp936') as fp:
    fp.write(s)

with open('sample.txt', encoding='cp936') as fp:
    print(fp.read())

# 示例9-2 文件编码转换复制 (CP936 -> UTF8)
def fileCopy(src, dst, srcEncoding, dstEncoding):
    with open(src, 'r', encoding=srcEncoding) as srcfp:
        with open(dst, 'w', encoding=dstEncoding) as dstfp:
            dstfp.write(srcfp.read())
fileCopy('sample.txt', 'sample_new.txt', 'cp936', 'utf8')

# 示例9-3 遍历并输出文件的所有行
with open('sample.txt', encoding='cp936') as fp:
    for line in fp:
        print(line, end='')

# 示例9-4 修改特定位置的字符
with open('sample.txt', 'r+', encoding='cp936') as fp:
    fp.seek(13)
    fp.write('测试')

# 示例9-6 统计文本文件中最长行的长度和内容
with open('sample.txt', encoding='cp936') as fp:
    result = [0, '']
    for line in fp:
        t = len(line)
        if t > result[0]:
            result = [t, line]
    print("\n最长行:", result)

# 示例9-7 使用json进行数据交换
import json
with open('test_json.txt', 'w') as fp:
    json.dump({'a':1, 'b':2, 'c':3}, fp)

with open('test_json.txt', 'r') as fp:
    print(json.load(fp))

# 示例9-8 使用csv模块
import csv
with open('test.csv', 'w', newline='') as fp:
    test_writer = csv.writer(fp, delimiter=' ', quotechar='"')
    test_writer.writerow(['red', 'blue', 'green'])
    test_writer.writerow(['test_string']*5)

with open('test.csv', newline='') as fp:
    test_reader = csv.reader(fp, delimiter=' ', quotechar='"')
    for row in test_reader:
        print(row)

## 9.3 二进制文件操作案例精选
Python中常用的序列化模块有 `struct`、`pickle`、`shelve`、`marshal`。

In [ ]:
# 9.3.2 使用struct模块读写二进制文件
import struct
n = 130000000
x = 96.45
b = True
s = 'a1@中国'

sn = struct.pack('if?', n, x, b) # i整数, f实数, ?逻辑值
with open('sample_struct.dat', 'wb') as f:
    f.write(sn)
    f.write(s.encode())

with open('sample_struct.dat', 'rb') as f:
    sn = f.read(9)
    tu = struct.unpack('if?', sn)
    n, x, b1 = tu
    print('n=', n, 'x=', x, 'b1=', b1)
    s_bytes = f.read(9)
    print('s=', s_bytes.decode())

# 9.3.1 使用pickle模块
import pickle
data = (123, 99.056, '中国人民', [1, 2, 3])
with open('sample_pickle.dat', 'wb') as f:
    pickle.dump(len(data), f)
    for item in data:
        pickle.dump(item, f)

with open('sample_pickle.dat', 'rb') as f:
    n = pickle.load(f)
    for _ in range(n):
        print(pickle.load(f))

# 9.3.3 使用shelve模块 (像操作字典一样)
import shelve
with shelve.open('shelve_test.dat') as fp:
    fp['zhangsan'] = {'age':38, 'sex':'Male', 'address':'SDIBT'}
    fp['lisi'] = {'age':40, 'sex':'Male', 'qq':'1234567'}

with shelve.open('shelve_test.dat') as fp:
    print(fp['zhangsan']['age']) # 38
    print(fp['lisi']['qq'])      # 1234567

### 9.3.4 其他常见类型二进制文件操作 (需安装扩展库)
* `xlwt` / `xlrd`: 处理低版本 Excel
* `openpyxl`: 处理 Excel 2007+ (.xlsx)
* `python-docx`: 处理 Word (.docx)

In [ ]:
# openpyxl 示例 (如果环境没有安装会报错, 仅展示代码逻辑)
"""
import openpyxl
from openpyxl import Workbook
fn = 'test.xlsx'
wb = Workbook()
ws = wb.create_sheet(title='你好,世界')
ws['A1'] = '这是第一个单元格'
ws['B1'] = 3.1415926
ws.append([1,2,3,4,5])
wb.save(fn)
"""